# GRU round 1 — positive-class weight comparison
## Goal
Compare **five positive weights** while keeping cleaned inputs, grouped folds, architecture, optimiser, seeds and 20 training epochs fixed. Each fit starts from scratch. No early stopping, oversampling or architecture search is used in this round.

The weights are `r ** alpha`, where `r = training negatives / training positives` and alpha is 0, 0.25, 0.5, 0.75 or 1. Alpha=0 is ordinary BCE; alpha=1 reproduces full weighting. Weights are computed within each training fold, across the pooled datasets.

**Selection:** highest mean fold trapezoidal PR AUC; ROC AUC breaks ties, then lower alpha. Five-fold CV gives 25 fits; optional final refit adds one. Fold SD describes variability, not a confidence interval. Reusing CV to select weights makes the winner's score exploratory, not an independent test estimate. Keep a separate test set for final confirmation.

**Inputs:** the same full cleaner outputs as your previous notebook. No data or trained weights are embedded here. Dataset 2 uses its existing binary `label`, with its conversion rule unchanged. Predictions are per transcript site, not per entire transcript.

**Quick start:** update Section 2 paths, run all cells, then save the notebook to retain displayed outputs. Later set `RUN_TRAINING=False` and set `EXISTING_RUN_DIR` to the printed run directory to regenerate charts without loading source data or retraining.

## 1. Environment

In [1]:
%pip install "numpy>=1.24,<3" "pandas>=2,<3" "scikit-learn>=1.3,<2" "matplotlib>=3.7,<4" "torch>=2.2,<3" "joblib>=1.3,<2" ipykernel

  Using cached numpy-2.5.4-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached pandas-2.3.3-cp314-cp314-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (91 kB)
  Using cached scikit_learn-1.9.1-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (9.3 kB)
  Using cached matplotlib-3.11.2-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (80 kB)
  Using cached torch-2.14.1-cp314-cp314-manylinux_2_28_x86_64.whl.metadata (37 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached pytz-2026.5-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2026.5-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached scipy-1.18.1-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached narwhals-2.27.1-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.7.0-py3-none-any.whl.metadata (24 kB)
  Using cached contourpy-1.4.0-cp314-cp314-manylin

In [1]:
import sys
import numpy as np
import pandas as pd
import sklearn
import torch

print("Python:", sys.version)
print("Interpreter:", sys.executable)
print("NumPy:", np.__version__)
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

Python: 3.12.15 (main, Oct  6 2026, 23:50:18) [GCC 15.3.0]
Interpreter: /home/ubuntu/miniforge3/envs/m6a-gru/bin/python
NumPy: 2.5.4
PyTorch: 2.14.1+cu130
GPU available: False


In [ ]:
# Uncomment once if packages are missing; restart the kernel afterwards.
# %pip install "numpy>=1.24,<3" "pandas>=2,<3" "scikit-learn>=1.3,<2" "matplotlib>=3.7,<4" "torch>=2.2,<3" "joblib>=1.3,<2" ipykernel
from pathlib import Path
from datetime import datetime, timezone
import gc, gzip, hashlib, importlib.metadata, json, random, re, shutil, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedGroupKFold, ParameterSampler
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_recall_curve,
    roc_curve, auc, precision_score, recall_score, f1_score, accuracy_score)
from IPython.display import display
plt.rcParams.update({'figure.figsize': (8, 4), 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False})
pd.set_option('display.max_columns', 20)
import time

## 2. Paths and fixed experiment settings
The Windows paths below come from your executed notebook. On Ronin, change PROJECT_ROOT and any input paths that differ. Output folders are timestamped and never overwrite an earlier run.
`REFIT_SELECTED=True` saves a final all-data model and complete prediction files; these predictions must not be treated as held-out evaluation. Fold weights and scalers are also saved.

In [ ]:
# Ronin example only — replace with your actual folder.
# PROJECT_ROOT = Path('/home/ubuntu/Team_Project')
PROJECT_ROOT = Path('/home/ubuntu/project')

DATASETS = [
    dict(dataset_id='dataset0', clean_dir='outputs/data_cleaning/dataset0/20261006T143139_734059Z',
         use_for_training=True, raw_json='dataset/data0/dataset0_json/dataset0.json'),
    dict(dataset_id='dataset1', clean_dir='outputs/data_cleaning/dataset1/20261006T144017_412380Z',
         use_for_training=True, raw_json='dataset/data1/dataset1_json/dataset1.json'),
    dict(dataset_id='dataset2', clean_dir='outputs/data_cleaning/dataset2/20261006T144639_349494Z',
         use_for_training=True, raw_json='dataset/data2/dataset2_json/dataset2.json'),
]
OUTPUT_ROOT = PROJECT_ROOT / 'outputs' / 'gru_class_weight_round1'
SEED = 42
N_FOLDS = 5
THRESHOLD = 0.5  # Only for precision/recall/F1; ROC/PR AUC use continuous scores.
PREDICT_BATCH_SIZE = 1024
QC_FALLBACK = 'training_prevalence'  # Or 'error'; review fallback audit before submission.
RUN_TRAINING = True
EXISTING_RUN_DIR = None  # e.g. Path('/home/ubuntu/project/outputs/gru_class_weight/2026...')
REFIT_SELECTED = True
ALPHAS = [0.0, 0.25, 0.5, 0.75, 1.0]
BASE_CONFIG = dict(hidden_size=32, dropout=0.2, lr=1e-3, weight_decay=1e-4,
                   batch_size=256, epochs=20, class_weighting=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def resolve_path(value):
    p = Path(value).expanduser()
    return (p if p.is_absolute() else PROJECT_ROOT / p).resolve()
def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
seed_everything(SEED)
print('Device:', DEVICE, '| PyTorch:', torch.__version__)
print('CV fits:', N_FOLDS * len(ALPHAS), '| optional final refit:', REFIT_SELECTED)

## 3. Load and validate the unchanged cleaned features
Scaling is still fitted only within training folds. Results-only mode skips input loading.

In [ ]:
LOCATIONS = ['minus1', 'center', 'plus1']
MEASURES = ['dwell', 'signal_sd', 'signal_mean']
STATS = ['mean', 'std', 'median', 'q25', 'q75']
# Explicit ordering is essential: do not reshape the CSV's original column order.
POSITION_COLUMNS = [[f'{loc}_{measure}_{stat}' for stat in STATS for measure in MEASURES]
                    for loc in LOCATIONS]
SUMMARY_COLUMNS = [c for cols in POSITION_COLUMNS for c in cols]
KEY = ['transcript_id', 'transcript_position']

def validate_features(frame):
    required = KEY + ['sequence', 'n_reads_valid'] + SUMMARY_COLUMNS
    missing = sorted(set(required) - set(frame.columns))
    if missing: raise ValueError(f'Missing feature columns: {missing}')
    if frame.empty: raise ValueError('No sites found.')
    if frame[KEY].isna().any().any() or frame.duplicated(KEY).any():
        raise ValueError('Missing or duplicate site keys.')
    if frame.transcript_id.astype(str).str.strip().eq('').any():
        raise ValueError('Empty transcript ID.')
    pos = pd.to_numeric(frame.transcript_position, errors='raise').to_numpy(dtype=float)
    if not np.isfinite(pos).all() or (pos < 0).any() or (pos != np.floor(pos)).any():
        raise ValueError('Positions must be nonnegative integers.')
    seq = frame.sequence.astype('string')
    if not seq.str.fullmatch('[ACGT]{7}').fillna(False).all():
        raise ValueError('Expected seven-base A/C/G/T sequences.')
    if not seq.str.slice(1, 6).str.fullmatch('[AGT][AG]AC[ACT]').fillna(False).all():
        raise ValueError('Non-DRACH central motif; inspect cleaning.')
    values = frame[SUMMARY_COLUMNS + ['n_reads_valid']].to_numpy(dtype=float)
    if not np.isfinite(values).all() or (frame.n_reads_valid < 1).any():
        raise ValueError('Nonfinite summaries or invalid coverage.')

In [ ]:
def read_clean_csv(path):
    if not path.is_file(): raise FileNotFoundError(f'Check input path: {path}')
    return pd.read_csv(path, dtype={'transcript_id': str, 'gene_id': str})

def load_inputs(configs):
    if not configs: raise ValueError('Add at least one dataset.')
    ids = [c['dataset_id'] for c in configs]
    if len(set(ids)) != len(ids) or any(not re.fullmatch(r'[A-Za-z0-9_-]+', x) for x in ids):
        raise ValueError('Dataset IDs must be unique and filename-safe.')
    if 'COMBINED' in ids: raise ValueError('COMBINED is reserved for the pooled result.')
    tables, train_parts, manifests, paths, summary = {}, [], {}, {}, []
    for c in configs:
        name = c['dataset_id']; folder = resolve_path(c['clean_dir'])
        manifest_path = folder / 'run_manifest.json'
        if not manifest_path.is_file(): raise FileNotFoundError(manifest_path)
        manifest = json.loads(manifest_path.read_text())
        if 'max_lines' not in manifest or manifest['max_lines'] is not None:
            raise ValueError(f'{name}: use a full cleaning run with max_lines=null.')
        if manifest.get('dataset_id') != name:
            raise ValueError(f'{name}: ID differs from cleaner manifest.')
        signal_path = resolve_path(c['prediction_file']) if c.get('prediction_file') else folder/'clean_signal_sites.csv.gz'
        signal = read_clean_csv(signal_path)
        validate_features(signal)
        signal['transcript_position'] = signal.transcript_position.astype('int64')
        if 'dataset_id' in signal and not signal.dataset_id.eq(name).all():
            raise ValueError(f'{name}: unexpected provenance in signal file.')
        signal['dataset_id'] = name
        signal['label'] = np.nan  # Prediction-only labels remain unavailable by design.
        train_count = 0
        train_path = None
        if c['use_for_training']:
            # Original label_type may be fraction: the exported binary label is authoritative.
            train_path = resolve_path(c['train_file']) if c.get('train_file') else folder/'clean_labeled_sites.csv.gz'
            labelled = read_clean_csv(train_path)
            if labelled.empty:
                raise ValueError(f'{name}: no binary-labelled rows in {train_path}. For fractional sources, rerun the cleaner with your chosen binary conversion rule and use that run folder.')
            validate_features(labelled)
            if 'label' not in labelled or not labelled.label.isin([0,1]).all():
                raise ValueError(f'{name}: need nonmissing binary labels, not fractions.')
            labelled['transcript_position'] = labelled.transcript_position.astype('int64')
            labelled['label'] = labelled.label.astype('int64')
            if 'gene_id' not in labelled: labelled['gene_id'] = pd.NA
            # Ensure training summaries are exactly the summaries used for prediction.
            check_cols = KEY + SUMMARY_COLUMNS + ['sequence','n_reads_valid']
            matched = labelled[check_cols].merge(signal[check_cols], on=KEY,
                        how='left', validate='one_to_one', suffixes=('_train','_signal'), indicator=True)
            if not matched['_merge'].eq('both').all():
                raise ValueError(f'{name}: labelled rows absent from prediction file.')
            for col in SUMMARY_COLUMNS + ['sequence','n_reads_valid']:
                if not matched[col+'_train'].eq(matched[col+'_signal']).all():
                    raise ValueError(f'{name}: training/prediction features differ in {col}.')
            signal = signal.drop(columns='label').merge(labelled[KEY+['label']], on=KEY,
                        how='left', validate='one_to_one', sort=False)
            labelled['dataset_id'] = name
            train_parts.append(labelled)
            train_count = len(labelled)
        raw_value = c.get('raw_json') or manifest.get('inputs', [{}])[0].get('path')
        if not raw_value: raise ValueError(f'{name}: supply raw_json for complete key checking.')
        raw_path = resolve_path(raw_value)
        if not raw_path.is_file(): raise FileNotFoundError(f'{name}: set raw_json to its current path: {raw_path}')
        tables[name] = signal.reset_index(drop=True)
        manifests[name] = manifest
        paths[name] = dict(signal=str(signal_path), training=str(train_path) if train_path else None,
                           manifest=str(manifest_path), raw_json=str(raw_path))
        summary.append(dict(dataset_id=name, prediction_sites=len(signal), training_sites=train_count,
                            use_for_training=c['use_for_training'], label_type=manifest.get('config',{}).get('label_type')))
    if not train_parts: raise ValueError('Enable at least one binary training dataset.')
    training = pd.concat(train_parts, ignore_index=True)
    if training.label.nunique() != 2: raise ValueError('Combined training needs both classes.')
    return tables, training, manifests, paths, pd.DataFrame(summary)


if RUN_TRAINING:
    if QC_FALLBACK not in ('training_prevalence', 'error'): raise ValueError('Unknown QC_FALLBACK.')
    if N_FOLDS < 2 or PREDICT_BATCH_SIZE < 1 or not 0 < THRESHOLD < 1:
        raise ValueError('Invalid run settings.')
    if any((p/'.git').exists() for p in (OUTPUT_ROOT.resolve(), *OUTPUT_ROOT.resolve().parents)):
        raise ValueError('Keep OUTPUT_ROOT outside the Git repository.')
    dataset_tables, train_df, cleaning_manifests, input_paths, input_summary = load_inputs(DATASETS)
    RUN_DIR = OUTPUT_ROOT / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    RUN_DIR.mkdir(parents=True, exist_ok=False)
    input_summary.to_csv(RUN_DIR/'input_summary.csv', index=False)
    display(input_summary)
    print(f'Combined training: {len(train_df):,} rows; positive fraction={train_df.label.mean():.2%}')
    print('Outputs:', RUN_DIR)
else:
    if EXISTING_RUN_DIR is None: raise ValueError('Set EXISTING_RUN_DIR for results-only mode.')
    RUN_DIR = Path(EXISTING_RUN_DIR).expanduser().resolve()
    if not (RUN_DIR/'leaderboard.csv').is_file(): raise FileNotFoundError('No completed leaderboard in this run.')

## 4. Freeze the shared gene/transcript folds
All copies of a transcript stay together across datasets. Missing gene annotations use transcript grouping; this cannot guarantee fully gene-disjoint evaluation for unmapped transcripts. Overlaps and label disagreements are audited.

In [ ]:
if RUN_TRAINING:
    known = train_df.loc[train_df.gene_id.notna() & train_df.gene_id.astype('string').str.strip().ne(''), ['transcript_id','gene_id']].drop_duplicates()
    if known.groupby('transcript_id').gene_id.nunique().gt(1).any():
        raise ValueError('A transcript maps to different genes; reconcile the reference annotations.')
    mapping = known.set_index('transcript_id').gene_id.to_dict()
    mapped = train_df.transcript_id.map(mapping)
    groups = pd.Series(['gene:'+g if pd.notna(g) else 'transcript:'+t
                       for t,g in zip(train_df.transcript_id, mapped)])
    if groups.nunique() < N_FOLDS: raise ValueError('Too few independent groups for configured folds.')
    grouping_summary = dict(known_gene_rows=int(mapped.notna().sum()),
                            transcript_fallback_rows=int(mapped.isna().sum()), groups=int(groups.nunique()))
    print('Grouping:', grouping_summary)
    overlaps = train_df.loc[train_df.duplicated(KEY, keep=False), ['dataset_id']+KEY+['label']]
    overlaps.to_csv(RUN_DIR/'cross_dataset_shared_sites.csv', index=False)
    conflicts = overlaps.groupby(KEY).label.nunique().gt(1).sum() if len(overlaps) else 0
    print(f'Shared-site rows: {len(overlaps):,}; shared sites with disagreeing labels: {conflicts:,}')
    cv = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    folds = list(cv.split(np.zeros(len(train_df)), train_df.label, groups=groups))
    fold_id = np.full(len(train_df), -1, dtype=int)
    fold_summary = []
    for number, (tr, va) in enumerate(folds, 1):
        if set(groups.iloc[tr]) & set(groups.iloc[va]): raise ValueError('Group leakage.')
        if set(train_df.iloc[tr].transcript_id) & set(train_df.iloc[va].transcript_id):
            raise ValueError('Transcript leakage.')
        for role, idx in [('train',tr),('validation',va)]:
            if train_df.iloc[idx].label.nunique() != 2:
                raise ValueError(f'Fold {number} {role} lacks a class. Review groups or reduce N_FOLDS.')
            fold_summary.append(dict(fold=number, role=role, sites=len(idx),
                                     positive_fraction=float(train_df.iloc[idx].label.mean())))
        if (fold_id[va] != -1).any(): raise ValueError('Repeated validation assignment.')
        fold_id[va] = number
    assert (fold_id > 0).all()
    assignments = train_df[['dataset_id']+KEY+['gene_id','label']].copy()
    assignments['cv_group'] = groups.to_numpy(); assignments['validation_fold'] = fold_id
    assignments.to_csv(RUN_DIR/'cv_fold_assignments.csv.gz', index=False)
    display(pd.DataFrame(fold_summary))

## 5. Unchanged input representation
45 signal summary features form a 3 × 15 sequence. Seven-base one-hot encoding and log read count enter the dense head.

In [ ]:
def raw_arrays(frame):
    sequence = np.stack([frame[cols].to_numpy(dtype=np.float64)
                         for cols in POSITION_COLUMNS], axis=1)
    base_index = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    indices = np.array([[base_index[b] for b in seq] for seq in frame.sequence])
    bases = np.eye(4, dtype=np.float32)[indices].reshape(len(frame), 28)
    coverage = np.log1p(frame[['n_reads_valid']].to_numpy(dtype=np.float64))
    return sequence, bases, coverage


def fit_preprocessing(frame):
    seq, _, coverage = raw_arrays(frame)
    return {'sequence_scaler': StandardScaler().fit(seq.reshape(len(frame), 45)),
            'coverage_scaler': StandardScaler().fit(coverage)}

def transform(frame, preprocessing):
    seq, bases, cov = raw_arrays(frame)
    seq = preprocessing['sequence_scaler'].transform(seq.reshape(len(frame), 45)).reshape(-1, 3, 15)
    aux = np.concatenate([bases, preprocessing['coverage_scaler'].transform(cov)], axis=1)
    seq, aux = seq.astype('float32'), aux.astype('float32')
    if not np.isfinite(seq).all() or not np.isfinite(aux).all():
        raise ValueError('Nonfinite transformed values; inspect extreme inputs.')
    return seq, aux

## 6. Training: change only positive weight
Validation metrics are logged every epoch for diagnosis, **not** for checkpoint selection. Every fold uses its final epoch. Unweighted validation BCE is comparable across weight settings; weighted training losses use different objectives and should not be ranked against one another.

In [ ]:
class SiteGRU(nn.Module):
    def __init__(self, hidden_size=32, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(input_size=15, hidden_size=hidden_size, batch_first=True)
        self.head = nn.Sequential(nn.Linear(hidden_size + 29, 32), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(32, 1))
    def forward(self, seq, aux):
        _, hidden = self.gru(seq)
        return self.head(torch.cat([hidden[-1], aux], dim=1)).squeeze(1)

def make_loader(seq, aux, batch_size, labels=None, shuffle=False, seed=SEED):
    tensors = [torch.from_numpy(seq), torch.from_numpy(aux)]
    if labels is not None: tensors.append(torch.from_numpy(labels))
    return DataLoader(TensorDataset(*tensors), batch_size=batch_size, shuffle=shuffle,
                      num_workers=0, generator=torch.Generator().manual_seed(seed))

@torch.no_grad()
def predict_scores(model, seq, aux, batch_size=512):
    model.eval()
    result = [torch.sigmoid(model(s.to(DEVICE), a.to(DEVICE))).cpu().numpy()
              for s, a in make_loader(seq, aux, batch_size)]
    return np.concatenate(result)

def fit_model(seq, aux, labels, config, seed=SEED, validation=None):
    seed_everything(seed)
    model = SiteGRU(config['hidden_size'], config['dropout']).to(DEVICE)
    positive_weight = float((labels == 0).sum() / (labels == 1).sum()) ** config['alpha']
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(positive_weight, device=DEVICE))
    optimizer = torch.optim.AdamW(model.parameters(), lr=config['lr'], weight_decay=config['weight_decay'])
    batches = make_loader(seq, aux, config['batch_size'], labels, shuffle=True, seed=seed)
    history = []
    for epoch in range(1, config['epochs'] + 1):
        model.train(); total_loss = 0.0
        for s, a, target in batches:
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(s.to(DEVICE), a.to(DEVICE)), target.to(DEVICE))
            if not torch.isfinite(loss): raise RuntimeError('Nonfinite loss.')
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item() * len(target)
        row = {'epoch': epoch, 'weighted_train_loss': total_loss / len(labels), 'pos_weight': positive_weight}
        if validation is not None:
            vs, va, vy = validation
            vp = predict_scores(model, vs, va, PREDICT_BATCH_SIZE)
            metrics = metric_row(vy, vp)
            row.update(val_PR_AUC=metrics['PR_AUC_trapezoidal'], val_ROC_AUC=metrics['ROC_AUC'])
            p = np.clip(vp.astype(float), 1e-7, 1-1e-7)
            row['val_unweighted_BCE'] = float(-(vy*np.log(p)+(1-vy)*np.log1p(-p)).mean())
        history.append(row)
        if epoch == 1 or epoch % 5 == 0 or epoch == config['epochs']:
            print(f'Epoch {epoch:02d}/{config["epochs"]}: loss={history[-1]["weighted_train_loss"]:.4f}', flush=True)
    return model, pd.DataFrame(history)

## 7. Evaluation and saved fold checkpoints
PR AUC uses trapezoidal integration; average precision is also retained. AUCs use continuous scores. Precision/recall/F1 use 0.5. Each out-of-fold row was excluded from that fold’s fitting.

In [ ]:
def metric_row(y, scores):
    y = np.asarray(y, dtype=int); scores = np.asarray(scores, dtype=float)
    if len(y) != len(scores) or not np.isfinite(scores).all() or ((scores<0)|(scores>1)).any():
        raise ValueError('Invalid scores or score/label length mismatch.')
    if not np.isin(y, [0,1]).all(): raise ValueError('Metrics require binary labels.')
    result = dict(n_scored_sites=len(y), positive_fraction=float(y.mean()) if len(y) else np.nan,
                  ROC_AUC=np.nan, PR_AUC_trapezoidal=np.nan, average_precision=np.nan,
                  precision=np.nan, recall=np.nan, F1=np.nan, accuracy=np.nan,
                  status='no binary labels')
    if not len(y): return result
    pred = (scores >= THRESHOLD).astype(int)
    result.update(precision=precision_score(y,pred,zero_division=0),
        recall=recall_score(y,pred,zero_division=0), F1=f1_score(y,pred,zero_division=0),
        accuracy=accuracy_score(y,pred))
    if len(np.unique(y)) < 2:
        result['status'] = 'one class; ROC/PR AUC unavailable'
        return result
    precision, recall, _ = precision_recall_curve(y, scores)
    result.update(ROC_AUC=roc_auc_score(y,scores), PR_AUC_trapezoidal=auc(recall,precision),
                  average_precision=average_precision_score(y,scores), status='ok')
    return result

def evaluate_tables(frames, scores_by_dataset, model_name, evaluation):
    records, pooled_y, pooled_scores = [], [], []
    for name, frame in frames.items():
        scores = np.asarray(scores_by_dataset[name])
        if len(scores) != len(frame): raise ValueError(f'{name}: score length mismatch.')
        mask = frame.label.notna().to_numpy()
        y = frame.loc[mask,'label'].to_numpy(dtype=int)
        s = scores[mask]
        records.append(dict(model=model_name, evaluation=evaluation, dataset_id=name,
                            n_prediction_sites=len(frame), **metric_row(y,s)))
        pooled_y.append(y); pooled_scores.append(s)
    records.append(dict(model=model_name, evaluation=evaluation, dataset_id='COMBINED',
                        n_prediction_sites=sum(map(len,frames.values())),
                        **metric_row(np.concatenate(pooled_y), np.concatenate(pooled_scores))))
    return pd.DataFrame(records)

def run_cv(config, experiment_name):
    folder = RUN_DIR / experiment_name; folder.mkdir(exist_ok=False)
    oof = np.full(len(train_df), np.nan); records = []
    for number,(tr,va) in enumerate(folds,1):
        print(f'\n{experiment_name}: fold {number}/{N_FOLDS}', flush=True)
        prep = fit_preprocessing(train_df.iloc[tr])
        train_seq, train_aux = transform(train_df.iloc[tr],prep)
        val_seq, val_aux = transform(train_df.iloc[va],prep)
        model,history = fit_model(train_seq,train_aux,
            train_df.iloc[tr].label.to_numpy(dtype=np.float32),config,seed=SEED+number, validation=(val_seq,val_aux,train_df.iloc[va].label.to_numpy(dtype=np.float32)))
        scores = predict_scores(model,val_seq,val_aux,PREDICT_BATCH_SIZE)
        oof[va] = scores
        records.append(dict(fold=number, **metric_row(train_df.iloc[va].label,scores)))
        history.to_csv(folder/f'fold_{number}_history.csv',index=False)
        torch.save({'state_dict': {k:v.detach().cpu() for k,v in model.state_dict().items()},
                    'config': config, 'architecture':'SiteGRU', 'fold':number,
                    'position_columns':POSITION_COLUMNS}, folder/f'fold_{number}_model.pt')
        joblib.dump(prep, folder/f'fold_{number}_preprocessing.joblib')
        del model,prep,train_seq,train_aux,val_seq,val_aux
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    if not np.isfinite(oof).all(): raise ValueError('Incomplete OOF scores.')
    fold_metrics = pd.DataFrame(records)
    fold_metrics.to_csv(folder/'fold_metrics.csv',index=False)
    output = assignments.copy(); output['score'] = oof
    output.to_csv(folder/'oof_predictions.csv.gz',index=False)
    # Include prediction-only datasets as explicit unavailable rows, not fabricated CV predictions.
    frames, by_dataset = {}, {}
    for c in DATASETS:
        name=c['dataset_id']; mask=train_df.dataset_id.eq(name).to_numpy()
        frames[name]=train_df.loc[mask].reset_index(drop=True)
        by_dataset[name]=oof[mask]
    table=evaluate_tables(frames,by_dataset,experiment_name,'grouped_OOF')
    table.to_csv(folder/'dataset_metrics.csv',index=False)
    objective=dict(PR_AUC_trapezoidal=float(fold_metrics.PR_AUC_trapezoidal.mean()),
                   ROC_AUC=float(fold_metrics.ROC_AUC.mean()))
    return dict(config=config.copy(),oof=oof,fold_metrics=fold_metrics,metrics=table,objective=objective)

## 8. Run the five comparisons
The alpha=1 candidate is the full-weight reference. The same fold seed controls initialisation and batch shuffling for each candidate. Small differences need confirmation with additional seeds; do not search for a favourable seed.

In [ ]:
if RUN_TRAINING:
    rows, fold_tables, dataset_metrics = [], [], []
    versions = {p: importlib.metadata.version(p) for p in ['numpy','pandas','scikit-learn','torch','matplotlib','joblib']}
    manifest = dict(seed=SEED, n_folds=N_FOLDS, alphas=ALPHAS, base_config=BASE_CONFIG,
                    inputs=input_paths, cleaner_manifests=cleaning_manifests,
                    grouping=grouping_summary, versions=versions, device=str(DEVICE),
                    interpretation='CV model selection; no independent test estimate')
    (RUN_DIR/'run_manifest.json').write_text(json.dumps(manifest,indent=2))
    for alpha in ALPHAS:
        name = f'alpha_{alpha:.2f}'
        config = dict(BASE_CONFIG, alpha=alpha)
        start = time.perf_counter()
        result = run_cv(config, name)
        fm = result['fold_metrics'].assign(candidate=name, alpha=alpha)
        fold_tables.append(fm)
        dataset_metrics.append(result['metrics'].assign(candidate=name, alpha=alpha))
        rows.append(dict(candidate=name, alpha=alpha,
                         mean_PR_AUC=fm.PR_AUC_trapezoidal.mean(), sd_PR_AUC=fm.PR_AUC_trapezoidal.std(),
                         mean_ROC_AUC=fm.ROC_AUC.mean(), sd_ROC_AUC=fm.ROC_AUC.std(),
                         mean_AP=fm.average_precision.mean(), seconds=time.perf_counter()-start))
        # Save completed progress after each candidate; partial runs are not auto-resumed.
        pd.DataFrame(rows).to_csv(RUN_DIR/'partial_leaderboard.csv',index=False)
    leaderboard = pd.DataFrame(rows).sort_values(['mean_PR_AUC','mean_ROC_AUC','alpha'],ascending=[False,False,True])
    leaderboard.to_csv(RUN_DIR/'leaderboard.csv',index=False)
    pd.concat(fold_tables).to_csv(RUN_DIR/'all_fold_metrics.csv',index=False)
    pd.concat(dataset_metrics).to_csv(RUN_DIR/'all_dataset_metrics.csv',index=False)

leaderboard = pd.read_csv(RUN_DIR/'leaderboard.csv')
fold_metrics = pd.read_csv(RUN_DIR/'all_fold_metrics.csv')
dataset_metrics = pd.read_csv(RUN_DIR/'all_dataset_metrics.csv')
selected = leaderboard.iloc[0]
print('Selected on validation:', selected.candidate, '| alpha:', selected.alpha)
display(leaderboard.round(5))
display(dataset_metrics[['candidate','dataset_id','n_scored_sites','positive_fraction','PR_AUC_trapezoidal','ROC_AUC']].round(4))
print('Run directory:', RUN_DIR)

## 9. Mean fold performance and paired changes
Error bars are ±1 fold standard deviation, not confidence intervals. The paired-delta plot compares each candidate with full weighting on the same fold. Pooled OOF AUC and mean fold AUC are different quantities.

In [ ]:
FIG_DIR = RUN_DIR/'figures'; FIG_DIR.mkdir(exist_ok=True)
ordered = leaderboard.sort_values('alpha')
names = ordered.candidate.tolist()
colors = dict(zip(names, ['#2457A7','#BD6C17','#647B36','#AE4D83','#52525B']))
def finish_figure(fig, name):
    fig.tight_layout()
    fig.savefig(FIG_DIR/f'{name}.png', dpi=160, bbox_inches='tight')
    plt.show()

fig, axes = plt.subplots(1,2,figsize=(12,4))
for ax, metric in zip(axes,['PR_AUC','ROC_AUC']):
    ax.errorbar(ordered.alpha, ordered['mean_'+metric], yerr=ordered['sd_'+metric], fmt='o-', capsize=4, color='#2457A7')
    ax.set(xlabel='Weight exponent alpha', ylabel=metric.replace('_',' '), title='Mean fold '+metric.replace('_',' ')+' ± fold SD')
    ax.set_xticks(ordered.alpha); ax.grid(alpha=.2)
finish_figure(fig,'01_mean_fold_metrics')

reference = fold_metrics.loc[fold_metrics.alpha.eq(1),['fold','PR_AUC_trapezoidal']].rename(columns={'PR_AUC_trapezoidal':'reference'})
paired = fold_metrics.merge(reference,on='fold',validate='many_to_one')
paired['delta_PR_AUC'] = paired.PR_AUC_trapezoidal-paired.reference
paired.to_csv(RUN_DIR/'paired_fold_deltas.csv',index=False)
fig, ax = plt.subplots(figsize=(8,4))
for fold, block in paired.groupby('fold'):
    block=block.sort_values('alpha')
    ax.plot(block.alpha,block.delta_PR_AUC,marker='o',label=f'Fold {fold}')
ax.axhline(0,color='black',ls='--',lw=1)
ax.set(xlabel='Weight exponent alpha',ylabel='PR AUC change versus alpha=1',title='Paired fold comparison'); ax.legend(ncol=3)
finish_figure(fig,'02_paired_deltas')

## 10. Per-dataset PR AUC and ROC AUC
Compare candidates within each dataset. PR AUC depends on positive prevalence: dataset 2’s high positive fraction makes its PR AUC unsuitable for direct comparison with dataset 0’s. Counts and prevalence are in the table above.

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(12,4))
for ax, metric in zip(axes,['PR_AUC_trapezoidal','ROC_AUC']):
    matrix=dataset_metrics.pivot(index='candidate',columns='dataset_id',values=metric).reindex(names)
    im=ax.imshow(matrix.to_numpy(),vmin=0,vmax=1,cmap='Blues',aspect='auto')
    ax.set_xticks(range(len(matrix.columns)),matrix.columns)
    ax.set_yticks(range(len(names)),names); ax.set_title('Pooled OOF '+metric)
    for i in range(len(matrix)):
        for j in range(len(matrix.columns)):
            v=matrix.iloc[i,j]
            ax.text(j,i,f'{v:.3f}' if np.isfinite(v) else 'N/A',ha='center',va='center',color='white' if v>.65 else 'black')
    fig.colorbar(im,ax=ax)
finish_figure(fig,'03_dataset_metrics')

## 11. OOF PR and ROC curves
Each panel uses identical rows across candidates. The PR dashed line is positive prevalence; the ROC dashed line is chance ranking. These curves combine predictions from five fitted models per candidate.

In [ ]:
oof = {name: pd.read_csv(RUN_DIR/name/'oof_predictions.csv.gz', dtype={'transcript_id':str}) for name in names}
reference_frame=oof[names[0]]
for frame in oof.values():
    assert frame[['dataset_id']+KEY+['label','validation_fold']].equals(reference_frame[['dataset_id']+KEY+['label','validation_fold']])
for dataset in [*reference_frame.dataset_id.unique(),'COMBINED']:
    mask = np.ones(len(reference_frame),dtype=bool) if dataset=='COMBINED' else reference_frame.dataset_id.eq(dataset).to_numpy()
    y=reference_frame.loc[mask,'label'].to_numpy()
    if len(np.unique(y))<2:
        print(dataset,': skipped curves (fewer than two classes)'); continue
    fig,axes=plt.subplots(1,2,figsize=(12,4))
    for name in names:
        p=oof[name].loc[mask,'score'].to_numpy()
        precision,recall,_=precision_recall_curve(y,p)
        fpr,tpr,_=roc_curve(y,p)
        axes[0].plot(recall,precision,color=colors[name],label=f'{name}: {auc(recall,precision):.4f}')
        axes[1].plot(fpr,tpr,color=colors[name],label=f'{name}: {roc_auc_score(y,p):.4f}')
    axes[0].axhline(y.mean(),color='black',ls='--',label=f'Prevalence {y.mean():.3f}')
    axes[1].plot([0,1],[0,1],'k--')
    axes[0].set(xlabel='Recall',ylabel='Precision',title=f'{dataset}: OOF PR (n={len(y):,})')
    axes[1].set(xlabel='False positive rate',ylabel='True positive rate',title=f'{dataset}: OOF ROC')
    for ax in axes: ax.set_xlim(0,1); ax.set_ylim(0,1.02); ax.legend(fontsize=8)
    finish_figure(fig,f'04_curves_{dataset}')

## 12. Epoch diagnostics
Curves average validation results across folds at each epoch. They diagnose learning but do not change the fixed final-epoch selection in this round. Do not compare weighted training loss magnitudes across different weights.

In [ ]:
histories=[]
for name in names:
    for path in sorted((RUN_DIR/name).glob('fold_*_history.csv')):
        histories.append(pd.read_csv(path).assign(candidate=name,fold=int(path.name.split('_')[1])))
history=pd.concat(histories,ignore_index=True)
history.to_csv(RUN_DIR/'all_epoch_history.csv',index=False)
fig,axes=plt.subplots(1,3,figsize=(15,4))
for name in names:
    h=history.loc[history.candidate.eq(name)].groupby('epoch').mean(numeric_only=True)
    for ax,metric in zip(axes,['val_PR_AUC','val_ROC_AUC','val_unweighted_BCE']):
        ax.plot(h.index,h[metric],label=name,color=colors[name]); ax.set(xlabel='Epoch',ylabel=metric,title='Mean fold '+metric)
axes[0].legend(fontsize=8)
finish_figure(fig,'05_epoch_diagnostics')

## 13. Optional all-data refit and complete prediction export
This fits only the selected alpha on all labelled rows, saves `model.pt`, `preprocessing.joblib`, and configuration, and exports original-key prediction CSVs. QC fallbacks are explicitly audited. No final training-set scores are used to select the winner.

In [ ]:
if RUN_TRAINING and REFIT_SELECTED:
    def unique_object(pairs):
        result = {}
        for key,value in pairs:
            if key in result: raise ValueError(f'Duplicate JSON key: {key}')
            result[key]=value
        return result

    def original_keys(path):
        opener=gzip.open if path.suffix=='.gz' else open
        keys=[]
        with opener(path,'rt',encoding='utf-8') as handle:
            for line_no,line in enumerate(handle,1):
                if not line.strip(): continue
                obj=json.loads(line,object_pairs_hook=unique_object)
                if not isinstance(obj,dict) or len(obj)!=1: raise ValueError(f'{path}: unexpected line {line_no}')
                transcript,positions=next(iter(obj.items()))
                if not isinstance(positions,dict) or len(positions)!=1: raise ValueError(f'{path}: invalid position object')
                position=next(iter(positions))
                if not re.fullmatch(r'\d+',position): raise ValueError('Invalid original position.')
                keys.append((transcript,position,int(position)))
        result=pd.DataFrame(keys,columns=['transcript_id','original_position','transcript_position'])
        if result.empty or result.duplicated(KEY).any(): raise ValueError(f'{path}: empty or duplicate original site keys.')
        return result

    raw_keys={}
    for c in DATASETS:
        name=c['dataset_id']; keys=original_keys(Path(input_paths[name]['raw_json']))
        if len(keys)!=cleaning_manifests[name]['raw_sites']:
            raise ValueError(f'{name}: raw input differs from cleaner raw-site count. Recheck paths.')
        check=dataset_tables[name][KEY].merge(keys[KEY],on=KEY,how='left',indicator=True,validate='one_to_one')
        if not check['_merge'].eq('both').all(): raise ValueError(f'{name}: clean keys absent from original data.')
        missing=len(keys)-len(dataset_tables[name])
        if missing and QC_FALLBACK=='error': raise ValueError(f'{name}: {missing} original sites lack clean features.')
        print(f'{name}: {len(keys):,} original sites; {missing:,} fallback sites')
        raw_keys[name]=keys

    def export_submission(name,clean_scores,folder):
        folder.mkdir(parents=True,exist_ok=True)
        clean=dataset_tables[name][KEY].copy(); clean['score']=np.asarray(clean_scores,dtype=float)
        merged=raw_keys[name].merge(clean,on=KEY,how='left',sort=False,validate='one_to_one')
        fallback=merged.score.isna()
        if fallback.any() and QC_FALLBACK=='error': raise ValueError('QC fallback disabled.')
        merged.loc[fallback,'score']=float(train_df.label.mean())
        if not np.isfinite(merged.score).all() or not merged.score.between(0,1).all():
            raise ValueError('Invalid submission scores.')
        submission=merged[['transcript_id','original_position','score']].rename(columns={'original_position':'transcript_position'})
        assert list(submission.columns)==['transcript_id','transcript_position','score']
        assert len(submission)==len(raw_keys[name])
        path=folder/f'{name}_predictions.csv'
        submission.to_csv(path,index=False,float_format='%.10g')
        audit=submission[KEY].copy(); audit['score_source']=np.where(fallback,'training_prevalence_fallback','GRU')
        audit.to_csv(folder/f'{name}_prediction_audit.csv',index=False)
        return dict(dataset_id=name,file=str(path),rows=len(submission),fallback_rows=int(fallback.sum()))

    def fit_all_and_evaluate(config,name):
        folder=RUN_DIR/name; folder.mkdir(exist_ok=True)
        prep=fit_preprocessing(train_df)
        seq,aux=transform(train_df,prep)
        model,history=fit_model(seq,aux,train_df.label.to_numpy(dtype=np.float32),config,seed=SEED)
        history.to_csv(folder/'all_data_history.csv',index=False)
        del seq,aux
        scores_by_dataset={}; exports=[]
        for dataset,frame in dataset_tables.items():
            seq,aux=transform(frame,prep)
            scores=predict_scores(model,seq,aux,PREDICT_BATCH_SIZE)
            scores_by_dataset[dataset]=scores
            exports.append(export_submission(dataset,scores,folder/'predictions'))
            del seq,aux
        metrics=evaluate_tables(dataset_tables,scores_by_dataset,name,'all_data_fit')
        metrics.to_csv(folder/'all_data_dataset_metrics.csv',index=False)
        checkpoint=dict(state_dict={k:v.detach().cpu() for k,v in model.state_dict().items()},
                        config=config,position_columns=POSITION_COLUMNS,
                        threshold=THRESHOLD,architecture='SiteGRU',training_prevalence=float(train_df.label.mean()))
        torch.save(checkpoint,folder/'model.pt')
        joblib.dump(prep,folder/'preprocessing.joblib')
        (folder/'config.json').write_text(json.dumps(config,indent=2))
        del model; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        return dict(metrics=metrics,scores=scores_by_dataset,exports=exports)
    selected_config = dict(BASE_CONFIG, alpha=float(selected.alpha))
    final_result = fit_all_and_evaluate(selected_config, 'selected')
    print('Final model and scalers:', RUN_DIR/'selected')
    print('Prediction files:', RUN_DIR/'selected'/'predictions')

## 14. Reload the saved final model without retraining
Run setup and function-definition cells first, or use a completed results-only run. Only load trusted checkpoint/joblib files. The example predicts clean sites; Section 13 includes original-key checking for full submission exports.

In [ ]:
# saved = RUN_DIR/'selected'
# checkpoint = torch.load(saved/'model.pt', map_location=DEVICE, weights_only=True)
# cfg = checkpoint['config']
# model = SiteGRU(cfg['hidden_size'], cfg['dropout']).to(DEVICE)
# model.load_state_dict(checkpoint['state_dict'])
# preprocessing = joblib.load(saved/'preprocessing.joblib')
# sites = pd.read_csv('/path/to/clean_signal_sites.csv.gz', dtype={'transcript_id':str})
# validate_features(sites)
# seq, aux = transform(sites, preprocessing)
# predictions = sites[KEY].copy()
# predictions['score'] = predict_scores(model, seq, aux, PREDICT_BATCH_SIZE)
# predictions.to_csv('/path/to/predictions.csv', index=False)

## Interpretation and next steps
- Inspect `leaderboard.csv` and paired fold differences before carrying a weight forward. A tiny mean gain is not proof of a robust improvement; repeat the strongest candidates with matching additional seeds.
- Inspect individual datasets as well as COMBINED. Pooled metrics give larger datasets more influence. Dataset 2 has a different label source and high prevalence; retain that context.
- Sigmoid scores from weighted loss are not necessarily calibrated probabilities. Changing weights can change both score ranking and the behaviour of a fixed 0.5 threshold.
- Keep architecture and data preparation fixed for this round. In the next round, test best-epoch selection or architecture changes using validation data, then confirm the complete selected procedure on independent test data.
- Saved outputs include fold assignments, fold checkpoints and scalers, OOF predictions, fold/dataset metrics, epoch histories, comparison figures and a run manifest. Save the notebook itself after running to keep embedded charts.

### Provenance and validation
Adapted from your supplied `m6a_gru_multiple_datasets_training(1).ipynb` and project handout. This is a new experiment, not a claim of improved performance. Real cleaned datasets were not supplied in this chat; full real-data execution must be performed in your environment. Do not report synthetic authoring checks as model results.

Authoring check: all five weights × five folds, optional final refit, prediction export, and results-only mode passed a small synthetic test with two epochs. Notebook structure and code syntax were validated. Synthetic outputs are deliberately excluded from this deliverable. Representative generated chart layout was inspected; real-data charts will be generated when you run it.